# Text Classification
## This notebook outlines the usage of NLP Feature extraction (CountVectorizer, TfidfVectorizer) in classification of text documents

### Import all the necessary libraries

In [7]:
from pprint import pprint
from time import time
import logging

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

### Choose a few categories for the entire 20 categories

In [9]:
# Load some categories from the training set
categories = [
    'alt.atheism',
    'talk.religion.misc',
]

In [10]:
print("Loading 20 newsgroups dataset for categories:")
print(categories)

Loading 20 newsgroups dataset for categories:
['alt.atheism', 'talk.religion.misc']


### Fetch documents for these 2 categories

In [12]:
data = fetch_20newsgroups(subset='train', categories=categories)
print(f"{len(data.filenames)} documents")
print(f"{len(data.target_names)} categories")
print()

857 documents
2 categories



### Define a pipeline combining a text feature extractor with a simple classifier

In [14]:
pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(tol=1e-3)),
])

### Specify parameter grid
- 'vect__max_df': (0.5, 0.75, 1.0)
- 'vect__max_features': (None, 5000, 10000, 50000)
- 'vect__ngram_range': ((1, 1), (1, 2))
- 'tfidf__use_idf': (True, False)
- 'tfidf__norm': ('l1', 'l2')
- 'clf__max_iter': (20,)
- 'clf__alpha': (0.00001, 0.000001)
- 'clf__penalty': ('l2', 'elasticnet')
- 'clf__max_iter': (10, 50, 80)

In [16]:
parameters = {
    'vect__max_df': (0.5, 0.75, 1.0),
    # 'vect__max_features': (None, 5000, 10000, 50000),
    'vect__ngram_range': ((1, 1), (1, 2)),  # unigrams or bigrams
    # 'tfidf__use_idf': (True, False),
    # 'tfidf__norm': ('l1', 'l2'),
    'clf__max_iter': (20,),
    'clf__alpha': (0.00001, 0.000001),
    'clf__penalty': ('l2', 'elasticnet'),
    # 'clf__max_iter': (10, 50, 80),
}

### Find the best parameters for both the feature extraction and the classifier

### Build a GridSearch with the pipeline and parameter grid

In [19]:
grid_search = GridSearchCV(pipeline, parameters, cv=5,
                           n_jobs=-1, verbose=1)

### Start the grid search

In [21]:
grid_search.fit(data.data, data.target)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect', CountVectorizer()),
                                       ('tfidf', TfidfTransformer()),
                                       ('clf', SGDClassifier())]),
             n_jobs=-1,
             param_grid={'clf__alpha': (1e-05, 1e-06), 'clf__max_iter': (20,),
                         'clf__penalty': ('l2', 'elasticnet'),
                         'vect__max_df': (0.5, 0.75, 1.0),
                         'vect__ngram_range': ((1, 1), (1, 2))},
             verbose=1)

### Best Score

In [23]:
print("Best score: %0.3f" % grid_search.best_score_)

Best score: 0.952


### Best Parameter

In [25]:
print("Best parameters set:")
best_parameters = grid_search.best_estimator_.get_params()
for param_name in sorted(parameters.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Best parameters set:
	clf__alpha: 1e-05
	clf__max_iter: 20
	clf__penalty: 'elasticnet'
	vect__max_df: 1.0
	vect__ngram_range: (1, 2)


### Choose the best model

In [27]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import nltk
from nltk.tokenize import word_tokenize
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import warnings

In [28]:
# 1️⃣ 讀取 2 Newsgroups 數據集
categories = [
    'alt.atheism',
    'talk.religion.misc',
]  # 選擇所有類別
newsgroups = fetch_20newsgroups(subset="all", categories=categories, remove=("headers", "footers", "quotes"))
X_texts = newsgroups.data  # 文本
y_labels = newsgroups.target  # 標籤

In [29]:
# 訓練集 & 測試集拆分
X_train, X_test, y_train, y_test = train_test_split(X_texts, y_labels, test_size=0.2, random_state=42)

In [30]:
# 2️⃣ 定義特徵提取方法
def get_count_vectorizer():
    return CountVectorizer(stop_words="english")

def get_tfidf_vectorizer():
    return TfidfVectorizer(stop_words="english")

In [31]:
# 3️⃣ 定義分類器
classifiers = {
    "Naïve Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=500),
    "SVM": SVC(kernel="linear"),
    "Decision Tree": DecisionTreeClassifier(),
    "clf": SGDClassifier(tol=1e-3),
}

# Feature Extractors:  Word2Vec

In [86]:
import numpy as np
from sklearn.base import TransformerMixin, BaseEstimator
from gensim.models import Word2Vec

class MeanEmbeddingVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = word2vec_model.vector_size
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in sentence.lower().split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for sentence in X
        ])

In [88]:
# 將訓練集文本進行簡單的分詞（這裡僅以空格切分）
tokenized_texts = [doc.lower().split() for doc in X_train]
w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=2, workers=4)

In [92]:
word2vec_extractor = MeanEmbeddingVectorizer(w2v_model.wv)

feature_extractors = {
    "CountVectorizer": get_count_vectorizer(),
    "TF-IDF": get_tfidf_vectorizer(),
    "Word2Vec": word2vec_extractor
}

In [98]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def get_param_grid(clf, vectorizer):
    grid = {}
    # 只有當向量器為 CountVectorizer 或 TfidfVectorizer 時，才加入向量器參數
    if isinstance(vectorizer, (CountVectorizer, TfidfVectorizer)):
        grid.update({
            'vect__max_df': (0.5, 0.75, 1.0),
            'vect__ngram_range': ((1, 1), (1, 2))
        })
    
    # 根據不同的分類器設定專屬參數
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    
    if isinstance(clf, MultinomialNB):
        grid['clf__alpha'] = (0.5, 1.0, 1.5)
        grid['clf__fit_prior'] = (True, False)
    elif isinstance(clf, LogisticRegression):
        grid['clf__C'] = (0.1, 1, 10)
        grid['clf__max_iter'] = (500,)
    elif isinstance(clf, SVC):
        grid['clf__C'] = (0.1, 1, 10)
    elif isinstance(clf, DecisionTreeClassifier):
        grid['clf__max_depth'] = (None, 10, 20, 30)
        grid['clf__min_samples_split'] = (2, 5, 10)
    # 其他分類器（例如 SGDClassifier）若有 max_iter 則調整
    elif hasattr(clf, "max_iter"):
        grid['clf__max_iter'] = (20,)
    
    # 若分類器有 alpha（但非 MultinomialNB）則調整
    if hasattr(clf, "alpha") and not isinstance(clf, MultinomialNB):
        grid['clf__alpha'] = (0.00001, 0.000001)
    # 若分類器有 penalty（且非 LogisticRegression）則調整
    if hasattr(clf, "penalty") and not isinstance(clf, LogisticRegression):
        grid['clf__penalty'] = ('l2', 'elasticnet')
    
    return grid


In [102]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

results = {}

for feat_name, vectorizer in feature_extractors.items():
    for clf_name, classifier in classifiers.items():
        # 檢查若使用 Word2Vec 特徵且分類器為 MultinomialNB 則跳過
        if feat_name == "Word2Vec" and clf_name == "Naïve Bayes":
            print(f"跳過組合：{feat_name} + {clf_name} (不相容)")
            continue

        pipeline = Pipeline([
            ("vect", vectorizer),
            ("clf", classifier)
        ])
        # 根據目前的分類器與向量器產生參數網格
        param_grid = get_param_grid(classifier, vectorizer)
        grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
        grid.fit(X_train, y_train)
        
        best_score = grid.best_score_
        best_params = grid.best_params_
        results[(feat_name, clf_name)] = (best_score, best_params)
        print(f"{feat_name} + {clf_name}: Best CV Accuracy = {best_score:.4f}")
        print("Best Parameters:", best_params)
        print("------")


CountVectorizer + Naïve Bayes: Best CV Accuracy = 0.7923
Best Parameters: {'clf__alpha': 0.5, 'clf__fit_prior': True, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 2)}
------
CountVectorizer + Logistic Regression: Best CV Accuracy = 0.7309
Best Parameters: {'clf__C': 10, 'clf__max_iter': 500, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 2)}
------
CountVectorizer + SVM: Best CV Accuracy = 0.7099
Best Parameters: {'clf__C': 0.1, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 1)}
------
CountVectorizer + Decision Tree: Best CV Accuracy = 0.6634
Best Parameters: {'clf__max_depth': 30, 'clf__min_samples_split': 10, 'vect__max_df': 1.0, 'vect__ngram_range': (1, 2)}
------


C:\Users\wayne\anaconda3\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:744: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


CountVectorizer + clf: Best CV Accuracy = 0.7152
Best Parameters: {'clf__alpha': 1e-05, 'clf__max_iter': 20, 'clf__penalty': 'l2', 'vect__max_df': 0.75, 'vect__ngram_range': (1, 1)}
------
TF-IDF + Naïve Bayes: Best CV Accuracy = 0.7739
Best Parameters: {'clf__alpha': 0.5, 'clf__fit_prior': False, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 1)}
------
TF-IDF + Logistic Regression: Best CV Accuracy = 0.7783
Best Parameters: {'clf__C': 10, 'clf__max_iter': 500, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 2)}
------
TF-IDF + SVM: Best CV Accuracy = 0.7853
Best Parameters: {'clf__C': 10, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 2)}
------
TF-IDF + Decision Tree: Best CV Accuracy = 0.6433
Best Parameters: {'clf__max_depth': 10, 'clf__min_samples_split': 2, 'vect__max_df': 0.75, 'vect__ngram_range': (1, 1)}
------
TF-IDF + clf: Best CV Accuracy = 0.7774
Best Parameters: {'clf__alpha': 1e-05, 'clf__max_iter': 20, 'clf__penalty': 'elasticnet', 'vect__max_df': 1.0, 'vect__ngram_range':

C:\Users\wayne\anaconda3\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:744: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [104]:
# 找出最佳的模型組合
(best_feat, best_clf), (best_cv_score, best_params) = max(results.items(), key=lambda x: x[1][0])
print("\nBest Model:")
print(f"best feat: {best_feat}")
print(f"best clf: {best_clf}")
print(f"best cv score: {best_cv_score:.4f}")
print("best params:", best_params)



Best Model:
best feat: CountVectorizer
best clf: Naïve Bayes
best cv score: 0.7923
best params: {'clf__alpha': 0.5, 'clf__fit_prior': True, 'vect__max_df': 0.5, 'vect__ngram_range': (1, 2)}


### Use the model to classify a piece of text

In [116]:
best_pipeline = GridSearchCV(
    Pipeline([
        ("vect", feature_extractors[best_feat]),
        ("clf", classifiers[best_clf])
    ]),
    get_param_grid(classifiers[best_clf], feature_extractors[best_feat]),
    cv=5, n_jobs=-1
)
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
print(f"test acc: {test_acc:.4f}")

test acc: 0.7902


In [118]:
sample_text = "This is a sample text to classify. It discusses computer graphics and rendering techniques."
predicted_class = best_pipeline.predict([sample_text])[0]

In [120]:
predicted_category = newsgroups.target_names[predicted_class]
print("Predicted class index:", predicted_class)
print("Predicted category:", predicted_category)

Predicted class index: 1
Predicted category: talk.religion.misc
